# 02 — Fine-tuning QLoRA do Llama no Colab (com checkpoints no Drive)

**Antes de começar**

1. Ambiente de execução → **Alterar tipo de ambiente de execução → GPU T4**.
2. Aceite a licença do modelo em <https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct> (e do 1B, se for usar).
3. Crie um token em <https://huggingface.co/settings/tokens> e salve nos **Secrets** do Colab com o nome `HF_TOKEN`.
4. Seu repositório: <https://github.com/nsouzajrs/desafio-tecnico-fase3>  (o código precisa estar lá antes de rodar este notebook).

**Como os checkpoints funcionam:** o treino salva o estado completo a cada 25 passos em uma pasta do seu Google Drive.
Se a sessão cair, **abra o notebook, rode as células 1 a 4 e depois a célula de treino de novo**: ela detecta o último checkpoint e continua de onde parou.

## 1. Conferindo a GPU

In [1]:
!nvidia-smi
import torch
print("CUDA disponível:", torch.cuda.is_available())
print("bf16 suportado:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None, "(a T4 não suporta; usamos fp16)")

Mon Nov 16 23:12:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Setup: Drive, repositório e dependências

In [2]:
# --- Setup do Colab: Drive + repositório + dependências -----------------------
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess, sys
REPO_URL = "https://github.com/nsouzajrs/desafio-tecnico-fase3"
REPO_DIR = "/content/desafio-tecnico-fase3"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
%cd {REPO_DIR}

# O PyTorch já vem instalado no Colab. Instalamos só o restante.
!pip install -q -r requirements-colab.txt

MessageError: Error: credential propagation was unsuccessful

## 3. Token do Hugging Face

In [ ]:
# Token do Hugging Face guardado nos "Secrets" do Colab (ícone de chave, na barra lateral).
# Crie o secret com o nome HF_TOKEN e ative "Acesso pelo notebook".
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("Token carregado:", bool(os.environ["HF_TOKEN"]))

## 4. Configuração do experimento

Todos os arquivos pesados vão para o **Drive**. Troque `RUN_NAME` para criar um novo experimento sem apagar o anterior.

In [ ]:
MODEL = "meta-llama/Llama-3.2-3B-Instruct"   # para um teste rápido: "meta-llama/Llama-3.2-1B-Instruct"
RUN_NAME = "run1_3b"
RUN_DIR = f"/content/drive/MyDrive/fase3/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)
print("Checkpoints em:", RUN_DIR)

## 5. Gerar o dataset (mesmo código do notebook 01)

In [ ]:
!python -m src.data.build_dataset --max-pubmed-train 500

## 6. Teste rápido (smoke test) — ~5 minutos

Antes de gastar uma hora de GPU, confirme que **tudo funciona** com poucos exemplos e poucos passos:
carrega o modelo, tokeniza, treina, salva. Se falhar, é aqui que você descobre.

In [ ]:
!python -m src.finetuning.train \
    --model {MODEL} \
    --output-dir /content/smoke_test \
    --max-train-samples 32 --epochs 1 --grad-accum 2 \
    --save-steps 5 --eval-steps 8 --logging-steps 1 --no-resume

**Se deu erro de memória (OOM):** reduza `--max-length` (ex.: 1024) ou troque para o modelo 1B.
**Se deu erro 401/403 ao baixar o modelo:** a licença do Llama ainda não foi aceita ou o token está errado.

## 7. Treino completo

Parâmetros que importam (e por quê):

| Parâmetro | Valor | Motivo |
|---|---|---|
| `--epochs 2` | 2 | com ~700 exemplos, 2 passadas bastam; mais que isso tende a decorar |
| `--batch-size 1` × `--grad-accum 8` | batch efetivo 8 | cabe na memória da T4 sem perder estabilidade |
| `--lr 2e-4` | 2e-4 | valor típico de QLoRA (só as matrizes LoRA são treinadas) |
| `--save-steps 25` | 25 | checkpoint frequente: a sessão gratuita pode cair a qualquer momento |

Tempo esperado na T4 com o 3B: algumas dezenas de minutos até cerca de 1–2 horas.

In [ ]:
!python -m src.finetuning.train \
    --model {MODEL} \
    --output-dir {RUN_DIR} \
    --epochs 2 --grad-accum 8 --lr 2e-4 \
    --save-steps 25 --eval-steps 50

> **Sessão caiu?** Rode as células 1 a 4 de novo e esta célula novamente. A linha
> `>>> Retomando do checkpoint: .../checkpoint-XX` confirma que continuou de onde parou.

## 8. Curva de aprendizado

In [ ]:
import json, matplotlib.pyplot as plt

summary = json.load(open(f"{RUN_DIR}/training_summary.json"))
log = summary["log_history"]
train = [(l["step"], l["loss"]) for l in log if "loss" in l]
evals = [(l["step"], l["eval_loss"]) for l in log if "eval_loss" in l]

plt.figure(figsize=(7, 4))
plt.plot(*zip(*train), label="treino")
if evals:
    plt.plot(*zip(*evals), marker="o", label="validação")
plt.xlabel("passo"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=.3)
plt.title("Curva de aprendizado")
plt.savefig(f"{RUN_DIR}/learning_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Loss final de treino:", summary["train_metrics"].get("train_loss"))

**Como interpretar:** a loss de treino deve cair. Se a loss de validação **sobe** enquanto a de treino continua caindo, é *overfitting*
(o modelo está decorando). Nesse caso, reduza as épocas.

## 9. O adapter final

In [ ]:
!ls -lh {RUN_DIR}/final_adapter
print("\nCopie esta pasta para artifacts/adapter no seu computador para usar o assistente localmente.")

O adapter tem poucas dezenas de MB, e é só ele que é preciso levar para a máquina local:
o modelo base é baixado do Hugging Face e o adapter é aplicado por cima.

**Próximo passo:** notebook 03 (avaliação base vs. fine-tuned).